# Simple RoA Example

### Purpose

This Notebook gives a miniaml example on how to use the RoA package. For a more detailed example on each step of the programm see the step by step examples in the notebooks directory

- [Extract and Persist Data](notebooks/01_Extract_and_Persist_Data.ipynb)
- [Construct disrupted Graph](notebooks/02_a_Construct_disrupted_Graph.ipynb)
- [(Optional) Visualize Graphs](notebooks/02_b_Visualize_Graphs.ipynb)
- [Robustness of Accessibility](notebooks/03_RobustnessOfAccessibility.ipynb)
- [AnalyzeResults](notebooks/04_AnalyzeResults.ipynb)

### Prerequisits

- Add Flooding data for all germany or copped to a area that matches or contains the relevant region
  - Default target dir is `css_geodata_service/robustness_of_accessibility/data/input/Flooding/HazardAreas`
- Required packages are defined in the pyproject.toml and can be easily installed using `uv`

### Dependencies
- To install dependencies run `uv sync` in the root directory of the project.
- If you are not familiar with `uv` just have a quick glance into the docs and install instructions of [astral](https://docs.astral.sh/uv/getting-started/installation/)
- If you don't want to use uv for testing you can manually manage and install your dependencies. This is not recommanded (and causes much more effort) and not accepted for production development


#### OMSNX
- Most of the processing of osm data is done using `osmnx`. There is a dedicated [github repository](https://github.com/gboeing/osmnx-examples) of the developer that contains a lot of examples on how to use `osmnx`. 

In [ ]:
from __future__ import annotations
from css_geodata_service.robustness_of_accessibility.examples.notebooks.notebook_utils import set_notbook_wd


set_notbook_wd()

In [ ]:
from css_geodata_service.robustness_of_accessibility.extract_osm import (
    extract_street_network,
    extract_poi,
    extract_boundaries,
)
from css_geodata_service.robustness_of_accessibility.disrupted_graph import (
    get_graph_affected_by_polygon_from_gdfs,
)
from css_geodata_service.robustness_of_accessibility.robustness_of_accessibility import (
    calculate_robustness_of_networks,
)
from shapely import Point, unary_union
import geopandas as gpd
from osmnx.convert import graph_to_gdfs
import osmnx as ox
from pathlib import Path
from css_geodata_service.robustness_of_accessibility.examples.notebooks.notebook_utils import (
    get_roa_hazard_data_path,
    set_notbook_wd,
)

In [ ]:
# ox.config(use_cache=True, log_console=True)
ox.settings.log_console = False
ox.settings.use_cache = True
ox.__version__

In [ ]:
# use name to get bounds
from css_geodata_service.robustness_of_accessibility.examples.notebooks.notebook_utils import RoaNotebookConfig


place_name = RoaNotebookConfig.place_name
place_gdf = ox.geocode_to_gdf(place_name)

In [ ]:
# 1. From the GeoDataFrame
if len(place_gdf) > 1:
    raise RuntimeError(f"gdf for {place_name} has multiple entries, could not safely deterine bounds")
boundary_geom = place_gdf.geometry.iloc[0]  # usually the first geometry

# 2. If it’s a MultiPolygon, merge the parts into one

if boundary_geom.geom_type == "MultiPolygon":
    boundary_geom = unary_union(boundary_geom)
elif boundary_geom.geom_type != "Polygon":
    raise RuntimeError(f"Could not combine boundary due to unexpected data structure {boundary_geom.geom_type}")
print(boundary_geom.bounds)
boundary_geom

In [ ]:
# get the street network of the region of interest (in this case as an undirected graph) and administrative boundaries.
print("2: extract network (orig), to_undirected(), administrative boundaries")
network_orig = extract_street_network(osm=None, library="osmnx", polygon=boundary_geom).to_undirected()
administrative_boundaries = extract_boundaries(osm=None, library="osmnx", polygon=boundary_geom)

In [ ]:
from css_geodata_service.robustness_of_accessibility.examples.notebooks.notebook_utils import RoaNotebookConfig


event = RoaNotebookConfig.event

In [ ]:
# data for trier can be downloaded via https://akrima-storage.dfki.uni-trier.de/dataset/osm-data-trier
# get the flooding area
print("3: get flooding area")
hazard_data_path: Path = get_roa_hazard_data_path(event=event)
# Import shapes of flooded areas
hazard_area = gpd.read_file(hazard_data_path)
hazrd_area_geoms = hazard_area.geometry

# Combine all shapes of the flooded are into one to make it easier to work with
hazard_area_multipolygon_df = gpd.GeoSeries(unary_union(hazrd_area_geoms))
hazard_area_multipolygon = hazard_area_multipolygon_df[0]

In [ ]:
# generate affected network
print("4: generate flood network")
nodes, edges = graph_to_gdfs(network_orig)
network_flood = get_graph_affected_by_polygon_from_gdfs(
    nodes,
    edges,
    hazard_area_multipolygon,
    exclude_bridges=True,
    undirected_graph=True,
)

In [ ]:
# identify positions of service providers
# (The sample of nodes to test the connectivity will be determined inside "calculate_robustness_of_networks".)
print("5: identify POIs...")
# clear data
fire_stations = None
hospitals = None

fire_stations = extract_poi(osm=None, library="osmnx", polygon=boundary_geom, amenities=["fire_station"])
hospitals = extract_poi(osm=None, library="osmnx", polygon=boundary_geom, amenities=["hospital"])
poi = {"fire_stations": fire_stations, "hospitals": hospitals}

In [ ]:
print(type(fire_stations))

In [ ]:
sample_distance_in_meters = 500

In [ ]:
robustness_values_multi_source = calculate_robustness_of_networks(
    samping_polygon=boundary_geom,
    street_network_original=network_orig,
    poi=poi,
    sample_boundaries=administrative_boundaries,
    undirected_graph=True,
    street_network_flooding=network_flood,
    sample_distance_in_meters=sample_distance_in_meters,
    use_multi_source_dijkstra=True,
)

In [ ]:
robustness_values_multi_source.keys()

In [ ]:
from css_geodata_service.robustness_of_accessibility.robustness_of_accessibility import calculate_roa_scores


scores = calculate_roa_scores(robustness_values_multi_source)
scores.keys()

In [ ]:
fire_station_scores = scores["fire_stations"]
fire_station_scores[fire_station_scores["score"] > 0]

# Visualization


In [ ]:
print(scores.keys())

In [ ]:
# select the type of service to visualize
# "hospitals", "fire_stations"
service_name_for_vis = "hospitals"
service_name_for_vis = "fire_stations"

In [ ]:
scores_to_use = scores[service_name_for_vis]
scores_to_use.head()

In [ ]:
from_locations = scores_to_use[["from_name", "from_position"]]
from_locations = from_locations.drop_duplicates()
from_locations

In [ ]:
def get_color(score_value: float):
    g = min(255.0, score_value * 510)
    r = min(255.0, (1 - score_value) * 510)
    b = 0
    return f"rgb({r},{g},{b})"


zoom_start = 12
icon_size = 12
legend_icon_size = 25
legend_text_size = 30

In [ ]:
import folium


def init_default_map(map_center: Point, flooded_area, tiles_list: list[str] | None = None, attr: str | None = None):
    location = [map_center.y, map_center.x]
    base_map = folium.Map(location=location, zoom_start=13, attr=attr, z_index_offset=250)

    flooding_group = folium.FeatureGroup(name="Flooding Area").add_to(base_map)

    # flooded_area.set_crs(epsg=4326, inplace=True)
    geo_json_flooded = folium.GeoJson(
        data=flooded_area,
        style_function=lambda x: {
            "fillColor": "#336699",
            "fillOpacity": 0.5,
            "color": "rgba(0, 0, 0, 0)",
        },
    )
    folium.Popup("Scenario 100 years").add_to(geo_json_flooded)
    geo_json_flooded.add_to(flooding_group)
    geo_json_flooded.add_to(flooding_group)

    return base_map

In [ ]:
import branca as branca

service_map = init_default_map(map_center=boundary_geom.centroid, flooded_area=hazard_area_multipolygon)

# add points to the map, with different colors depending on the score

score_group = folium.FeatureGroup(name="Score").add_to(service_map)


for idx, row in scores_to_use.iterrows():
    score = row["score"]
    # print(row)
    # break
    # color = cmap(norm(score))
    color = get_color(score)
    location = (row.to_centroid.y, row.to_centroid.x)
    folium.CircleMarker(location=location, radius=3, color=color, fill=True, fill_color=color).add_to(score_group)

hospital_group = folium.FeatureGroup(name="Hospitals").add_to(service_map)

for idx, row in from_locations.iterrows():
    location = (row.from_position.y, row.from_position.x)
    icon = folium.features.CustomIcon(
        f"css_geodata_service/robustness_of_accessibility/data/assets/icons/{service_name_for_vis}.png",
        icon_size=(icon_size * 3, icon_size * 3),
    )
    folium.Marker(location=location, icon=icon, popup=row["from_name"], z_index_offset=1000).add_to(hospital_group)


# specify the min and max values of your data
colormap = branca.colormap.LinearColormap(colors=["#00FF00", "#FF0000"], index=[0, 1], vmin=0, vmax=1)
# colormap = colormap.to_step(index=[0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1])
colormap.caption = f"{service_name_for_vis} Score"
colormap.add_to(service_map)


folium.LayerControl().add_to(service_map)

service_map.save(f"css_geodata_service/robustness_of_accessibility/data/output/map_{service_name_for_vis}.html")
service_map

In [ ]:
from datetime import datetime


now = datetime.now()
print(f"Done - {now}")